In [1]:
import ollama

import re
import json

In [2]:
def clean_code(code: str) -> str:
    """Remove comments and normalize"""
    code = re.sub(r'#.*$', '', code, flags=re.MULTILINE)
    code = '\n'.join(line for line in code.split('\n') if line.strip())
    return code.strip()

def rewrite(code: str, n_samples: int = 4) -> List[str]:
    """Generate n rewritten versions of the code"""
    
    cleaned_code = clean_code(code)
    
    prompt = f"""### Code:
{cleaned_code}

### Instruction:
 Please explain the functionality of the given code, then rewrite it in a single markdown code block. No additional clarifications.
"""
    
    rewrites = []
    for i in range(n_samples):
        try:
            response = ollama.chat(
                model="codellama:7b-instruct",
                messages=[{'role': 'user', 'content': prompt}],
                options={
                    'temperature': 0.8,
                    'top_p': 0.95,
                    'seed': None
                }
            )
            
            # Extract code block from response
            content = response['message']['content']
            code_blocks = re.findall(r'```\w*\n(.*?)```', content, re.DOTALL)
            
            if code_blocks:
                rewrite = clean_code(code_blocks[0])
                rewrites.append(rewrite)
            else:
                print(f"Failed to parse response: \n{content}\n")
        
        except Exception as e:
            print(f"Rewrite {i} failed: {e}")
            continue

    return rewrites

In [3]:
"""Take first element of detection dataset"""
with open("data/detection_data.json") as data_file:
    data = json.load(data_file)
    code = data[0]["code"]

code

'def find_missing(arr):\n    for i in range(len(arr) - 1):\n        if arr[i + 1] - arr[i] != 1:\n            return arr[i] + 1\n    return arr[-1] + 1\n'

In [4]:
rewrites = rewrite(code)
rewrites

['def find_missing(arr):\n    for i in range(len(arr) - 1):\n        if arr[i + 1] - arr[i] != 1:\n            return arr[i] + 1\n    return arr[-1] + 1',
 'def find_missing(arr):\n    if len(arr) > 0:\n        for i in range(len(arr) - 1):\n            if arr[i + 1] - arr[i] != 1:\n                return arr[i] + 1\n    return arr[-1] + 1',
 'def find_missing(arr):\n    for i in range(len(arr) - 1):\n        if arr[i + 1] - arr[i] != 1:\n            return arr[i] + 1\n    return arr[-1] + 1',
 'def find_missing(arr):\n    for i in range(len(arr) - 1):\n        if arr[i + 1] - arr[i] != 1:\n            return arr[i] + 1\n    return arr[-1] + 1']

In [5]:
len(rewrites)

4

In [8]:
i = 0
data_with_rewrites = []
for elem in data:
    elem['rewrites'] = rewrite(elem['code'], 4)
    elem['code'] = clean_code(elem['code'])
    data_with_rewrites.append(elem)
    i += 1
    print(f"Generated {i} of {len(data)}")
        

Generated 1 of 200
Generated 2 of 200
Generated 3 of 200
Generated 4 of 200
Failed to parse response: 

The given code defines a function `is_one_less_than_twice(num)` that takes an integer `num` as input and returns `True` if `num` is one less than twice its reverse, and `False` otherwise.

For example, if `num = 1234`, then the reverse of `num` is `4321`. If we subtract 1 from `4321`, we get `4320`, which is not equal to `1234`. Therefore, the function returns `False`.

On the other hand, if `num = 1235`, then the reverse of `num` is `5321`. If we subtract 1 from `5321`, we get `5320`, which is equal to `1235`. Therefore, the function returns `True`.

The code uses a technique called "string slicing" to reverse the digits of `num` and subtract 1. The syntax for string slicing is `[start:stop:step]`, where `start` and `stop` are the indices of the first and last characters in the slice, respectively, and `step` is the step size between each character in the slice. In this case, we spe

In [9]:
with open("detector/rewritten_data.json", "w") as data_file:
    json.dump(data_with_rewrites, data_file, indent=2)